# Nano_ext — インタラクティブ解析ノートブック

このノートブックは、ナノポア電流トレースからイベントを検出するための対話型UIを提供します。

## セットアップ

初回のみ以下を実行してください:

```bash
conda activate nano_ext
pip install "nano_ext[notebook]"
jupyter notebook
```

## 使い方

1. **セルを上から順に実行**します（Shift+Enter）
2. UIが表示されたら、各セクションを順に操作してください:
   - **Section 1**: 解析するファイルのパスを入力 → `Load` ボタンを押す
   - **Section 2**: フィルタ設定（`Auto-tune` を ON にするとノイズから自動設定）
   - **Section 3**: 検出パラメータの調整
   - **Section 4**: `Run Analysis` ボタンを押して解析を実行
   - **Section 5**: 結果の確認とCSVダウンロード

---

In [ ]:
# 必要なパッケージの確認
try:
    import nano_ext
    import ipywidgets
    import plotly
    print(f"nano_ext  {nano_ext.__version__}")
    print(f"ipywidgets {ipywidgets.__version__}")
    print(f"plotly    {plotly.__version__}")
    print("✓ すべてのパッケージが揃っています")
except ImportError as e:
    print(f"パッケージが不足しています: {e}")
    print('インストール: pip install "nano_ext[notebook]"')

In [ ]:
from nano_ext.outputs.notebook_widgets import NanoExtUI

ui = NanoExtUI()
ui.display()

---

## 解析後のカスタム処理

解析実行後、`ui._result` から結果オブジェクトに直接アクセスできます。

In [ ]:
# 解析後にこのセルを実行すると、イベントデータをDataFrameで取得できます
import pandas as pd

result = ui._result
if result is None:
    print("まだ解析が実行されていません。上のUIでRun Analysisを押してください。")
else:
    sr = result.signal_data.sampling_rate
    records = [
        {
            "event_id": ev.event_id,
            "start_sec": ev.start_idx / sr,
            "end_sec": ev.end_idx / sr,
            "duration_ms": ev.duration * 1000,
            "depth": ev.depth,
            "area": ev.area,
            "n_sublevels": len(ev.sublevels),
            "direction": ev.direction.value,
        }
        for ev in result.events
    ]
    df = pd.DataFrame(records)
    print(f"{len(df)} イベント検出")
    df.head()

In [ ]:
# 深さのヒストグラム（例）
if result is not None and result.events:
    import plotly.express as px
    fig = px.histogram(
        df, x="depth",
        nbins=50,
        title="イベント深さの分布",
        labels={"depth": f"Depth ({result.signal_data.units})"},
    )
    fig.show()

In [ ]:
# 持続時間のヒストグラム（例）
if result is not None and result.events:
    fig = px.histogram(
        df, x="duration_ms",
        nbins=50,
        title="イベント持続時間の分布",
        labels={"duration_ms": "Duration (ms)"},
    )
    fig.show()